In [1]:
# 10_decision_sweep.ipynb
# =============================================================================
# A deeper, fairer decision analysis addressing three concerns: (i) the test base
# rate (65% positive) is unrealistic for bankruptcy, so a single operating point
# is misleading; (ii) a single cost ratio is arbitrary; (iii) comparing raw Brier
# scores of uncalibrated models is unfair.
#
# We therefore (1) ISOTONIC-RECALIBRATE both models on validation data and report
# Brier before/after; (2) sweep the misclassification COST RATIO; and (3) sweep
# the deployment BASE RATE by combining each model's prevalence-independent
# operating characteristics (FNR, FPR on the test set) with a target prevalence,
# reporting the minimum expected cost and the relational model's relative saving
# across realistic base rates (1%-20%), not only the dataset's 65%.
# Probabilities are averaged over 5 seeds. Figures grayscale, dpi 600, png + pdf.
# =============================================================================
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
from torch_geometric.nn import HeteroConv, SAGEConv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[0]
PROCESSED = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "savefig.bbox": "tight",
                     "font.size": 10})
sns.set_theme(style="whitegrid", palette="Greys_r")
SEEDS = list(range(5))


def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)


data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
comp = data["company"]
Xc = comp.x.numpy().astype(np.float64); Xc[:, :2] = np.log1p(np.clip(Xc[:, :2], 0, None))
Xc = StandardScaler().fit(Xc[comp.train_mask.numpy()]).transform(Xc).astype(np.float32)
comp.x = torch.tensor(Xc)
y = comp.y
trm, vam, tem = comp.train_mask, comp.val_mask, comp.test_mask
EDGE_TYPES = [et for et in data.edge_types if int(et[1].split("_")[1]) <= 11]
ei_dict = {et: data[et].edge_index for et in EDGE_TYPES}
IN_DIMS = {"company": comp.x.size(1), "person": data["person"].x.size(1)}


class HeteroGNN(nn.Module):
    def __init__(self, hidden=64, out=2, layers=2, dropout=0.3):
        super().__init__()
        self.drop = dropout
        self.lin_in = nn.ModuleDict({nt: nn.Linear(d, hidden) for nt, d in IN_DIMS.items()})
        self.convs = nn.ModuleList([HeteroConv({et: SAGEConv((hidden, hidden), hidden)
                     for et in EDGE_TYPES}, aggr="sum") for _ in range(layers)])
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, out))

    def forward(self, xd, eid):
        h = {nt: F.relu(self.lin_in[nt](x)) for nt, x in xd.items()}
        for conv in self.convs:
            h = conv(h, eid)
            h = {nt: F.dropout(F.relu(v), p=self.drop, training=self.training) for nt, v in h.items()}
        return self.head(h["company"])


class IntraMLP(nn.Module):
    def __init__(self, d, hidden=64, out=2, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, out))

    def forward(self, x):
        return self.net(x)


def class_w():
    cls = torch.bincount(y[trm], minlength=2).float()
    return cls.sum() / (2 * cls.clamp(min=1))


def train_probs(kind, seed):
    torch.manual_seed(seed); np.random.seed(seed); w = class_w()
    if kind == "contagion":
        model = HeteroGNN(); xd = {nt: data[nt].x for nt in data.node_types}
        fwd = lambda: model(xd, ei_dict)
    else:
        model = IntraMLP(comp.x.size(1)); fwd = lambda: model(comp.x)
    opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=5e-4)
    best_val, best_state, wait = -1, None, 0
    for _ in range(200):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(fwd()[trm], y[trm], weight=w); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p = F.softmax(fwd(), 1)[:, 1].numpy()
        va = roc_auc_score(y[vam].numpy(), p[vam.numpy()])
        if va > best_val:
            best_val, best_state, wait = va, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= 30:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        p = F.softmax(fwd(), 1)[:, 1].numpy()
    return p[vam.numpy()], p[tem.numpy()]


yv = y[vam.numpy()].numpy(); yt = y[tem.numpy()].numpy()
probs = {}
for kind in ["intra", "contagion"]:
    pv = np.mean([train_probs(kind, s)[0] for s in SEEDS], axis=0)
    pt = np.mean([train_probs(kind, s)[1] for s in SEEDS], axis=0)
    probs[kind] = (pv, pt)

# --- (1) Isotonic recalibration (fit on validation) --------------------------
cal_rows = []
cal_probs = {}
for kind in ["intra", "contagion"]:
    pv, pt = probs[kind]
    iso = IsotonicRegression(out_of_bounds="clip").fit(pv, yv)
    pt_cal = iso.predict(pt)
    cal_probs[kind] = pt_cal
    cal_rows.append({"model": kind,
                     "Brier_raw": round(brier_score_loss(yt, pt), 4),
                     "Brier_isotonic": round(brier_score_loss(yt, pt_cal), 4),
                     "AUC": round(roc_auc_score(yt, pt), 4)})
cal_df = pd.DataFrame(cal_rows)
cal_df.to_csv(TAB / "10_recalibration.csv", index=False)
print("=== (1) Isotonic recalibration (Brier; lower is better) ===")
print(cal_df.to_string(index=False))

# --- Operating characteristics (prevalence-independent) ----------------------
def roc_pts(p):
    fpr, tpr, thr = roc_curve(yt, p)
    return fpr, tpr, thr   # FNR = 1 - tpr


# --- (2) Expected cost vs cost ratio at a realistic base rate pi -------------
def min_expected_cost(p, pi, cratio):
    fpr, tpr, _ = roc_curve(yt, p)
    fnr = 1 - tpr
    ec = pi * cratio * fnr + (1 - pi) * fpr
    return float(ec.min())


PI_REAL = 0.05
ratios = np.arange(1, 21)
cost_rows = []
fig, ax = plt.subplots(figsize=(5.0, 3.5))
for kind, col in [("intra", "#999999"), ("contagion", "#222222")]:
    ec = [min_expected_cost(cal_probs[kind], PI_REAL, r) for r in ratios]
    ax.plot(ratios, ec, color=col, linewidth=1.5, label=kind, marker="o", markersize=3)
    for r, e in zip(ratios, ec):
        cost_rows.append({"model": kind, "base_rate": PI_REAL, "cost_ratio": int(r),
                          "min_expected_cost": round(e, 5)})
ax.set_xlabel("Cost ratio $C_{FN}:C_{FP}$"); ax.set_ylabel(f"Min expected cost (base rate {PI_REAL:.0%})")
ax.legend(frameon=True, fontsize=8)
savefig(fig, "fig_cost_vs_ratio")
pd.DataFrame(cost_rows).to_csv(TAB / "10_cost_vs_ratio.csv", index=False)

# --- (3) Relative saving vs base rate (at fixed cost ratio 5:1) --------------
pis = np.linspace(0.02, 0.65, 40)
CR = 5
rel_rows = []
fig, ax = plt.subplots(figsize=(5.0, 3.5))
saving = []
for pi in pis:
    ec_i = min_expected_cost(cal_probs["intra"], pi, CR)
    ec_c = min_expected_cost(cal_probs["contagion"], pi, CR)
    rel = (ec_i - ec_c) / ec_i if ec_i > 0 else 0.0
    saving.append(100 * rel)
    rel_rows.append({"base_rate": round(float(pi), 4), "cost_ratio": CR,
                     "intra_cost": round(ec_i, 5), "contagion_cost": round(ec_c, 5),
                     "relative_saving_pct": round(100 * rel, 3)})
ax.plot(pis, saving, color="#222222", linewidth=1.6)
ax.axhline(0, color="black", linestyle=":", linewidth=0.8)
ax.axvline(PI_REAL, color="#777777", linestyle="--", linewidth=0.8)
ax.set_xlabel("Deployment base rate (bankruptcy prevalence)")
ax.set_ylabel(f"Relative expected-cost saving (%), $C_{{FN}}:C_{{FP}}$={CR}:1")
savefig(fig, "fig_cost_vs_baserate")
rel_df = pd.DataFrame(rel_rows)
rel_df.to_csv(TAB / "10_cost_vs_baserate.csv", index=False)

at5 = rel_df.iloc[(rel_df["base_rate"] - PI_REAL).abs().argmin()]
print(f"\n=== (2)-(3) Decision sweep (isotonic-calibrated) ===")
print(f"  at realistic base rate {PI_REAL:.0%}, cost ratio {CR}:1: "
      f"relative expected-cost saving = {at5['relative_saving_pct']:.1f}%")
print(f"  saving stays positive across base rates "
      f"{rel_df['base_rate'].min():.0%}-{rel_df['base_rate'].max():.0%}: "
      f"{'yes' if (rel_df['relative_saving_pct']>0).all() else 'no'} "
      f"(min {rel_df['relative_saving_pct'].min():.1f}%, max {rel_df['relative_saving_pct'].max():.1f}%)")

print("\n10_decision_sweep complete: recalibration + cost-ratio + base-rate sweeps; "
      "2 figures, 3 tables.")


=== (1) Isotonic recalibration (Brier; lower is better) ===
    model  Brier_raw  Brier_isotonic    AUC
    intra     0.2472          0.2374 0.6870
contagion     0.2109          0.1955 0.7804



=== (2)-(3) Decision sweep (isotonic-calibrated) ===
  at realistic base rate 5%, cost ratio 5:1: relative expected-cost saving = 13.9%
  saving stays positive across base rates 2%-65%: yes (min 0.6%, max 21.2%)

10_decision_sweep complete: recalibration + cost-ratio + base-rate sweeps; 2 figures, 3 tables.
